# API Project - Bus Bunching
### Multi-agency Comparsion (WMATA, MBTA, MTA)
Question: What route and temporal factors are associated with bus bunching on high-frequency corridors, and do these relationships vary across different transit agencies (NYC, Boston, and DC)?

WMATA
- High Frequency Routes: D40, D60, D80, C53, C61, D20, P40
- Low Frequency Routes: C81, P16, D24, P14, C23, P20, C35

MBTA
- High Frequency Routes: 1, 23, 28, 39, 57, 66, 111
- Low Frequency Routes: 64, 201, 50, 42, 87, 85, 70

MTA
- High Frequency Routes: B44, B46, B41, BX12, M14A+, M23+, Q98
- Low Frequency Routes: B32, M21, BX42, M50, Q67, B84, Q42

 Note: I added low frequency routes

In [1]:
import pandas as pd
import numpy as np
import requests
import ratelim
import tenacity
import time
import os

In [8]:
WMATA_KEY = "5ec08fcd21744590b1a0f655710c4c91"
MBTA_KEY = '4aa18b9edd1341e383dc8fe55a788cca'
MTA_KEY = '7027c575-3c63-477c-91d8-3d404472e8a0'

wmata_url = "https://api.wmata.com/Bus.svc/json/jBusPositions"
mbta_url = "https://api-v3.mbta.com/vehicles"
mta_url = "http://bustime.mta.info/api/siri/vehicle-monitoring.json"

wmata_headers = {"api_key": WMATA_KEY}
mbta_headers = {"x-api-key": MBTA_KEY}

In [4]:
wmata_routes = ['D40', 'D60', 'D80', 'C53', 'C61', 'D20', 'P40', 'C81', 'P16', 'D24', 'P14', 'C23', 'P20', 'C35']
mbta_routes = ['1', '23', '28', '39', '57', '66', '111', '64', '201', '50', '42', '87', '85', '70']
mta_routes = ['B44', 'B46', 'B41', 'BX12', 'M14A+', 'M23+', 'Q98', 'B32', 'M21', 'BX42', 'M50', 'Q67', 'B84', 'Q42']

In [5]:
wmata_fields = {
    'VehicleID': 'vehicle_id',
    'Lat': 'lat',
    'Lon': 'lon',
    'Deviation': 'deviation',
    'DateTime': 'timestamp',
    'TripID': 'trip_id',
    'RouteID': 'route_id',
    'DirectionNum': 'direction',
    'TripStartTime': 'trip_start_time'
}

In [ ]:
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)  # creating data folder to store csv

wmata_file = os.path.join(output_dir, "wmata.csv")
mbta_file = os.path.join(output_dir, "mbta.csv")
mta_file = os.path.join(output_dir, "mta.csv")

poll_count = 0  # tracking number of polls for auto push to githud

while True:
    try:
# WMATA
        try: #this allows for the loop to contuine if the pull fails for one agency
            wmata_dfs = []
            for route in wmata_routes:
                wmata_response = requests.get(wmata_url, params={"RouteID": route}, headers=wmata_headers)
                wmata_positions = wmata_response.json()['BusPositions']
                if not wmata_positions:  # skip if no active vehicles
                    continue
                df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
                df['agency'] = 'WMATA'
                df['timestamp'] = pd.to_datetime(df['timestamp'])
                df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
                wmata_dfs.append(df)
            if wmata_dfs:
                wmata_df = pd.concat(wmata_dfs, ignore_index=True)
                wmata_df.to_csv(wmata_file, mode='a', header=not os.path.exists(wmata_file), index=False) # appending pulls to exsiting file
        except Exception as e:
            print(f"WMATA error: {e}") #if an error this will print it and move on

# MBTA
        try:
            mbta_dfs = []
            for route in mbta_routes:
                mbta_response = requests.get(mbta_url, params={"filter[route]": route}, headers=mbta_headers)
                mbta_vehicles = mbta_response.json().get('data', [])  # empty list if no active vehicles
                if not mbta_vehicles:  # skip if no active vehicles
                    continue
                mbta_fields = []
                for vehicle in mbta_vehicles:
                    row = {
                        'vehicle_id': vehicle['id'],
                        'lat': vehicle['attributes']['latitude'],
                        'lon': vehicle['attributes']['longitude'],
                        'direction': vehicle['attributes']['direction_id'],
                        'timestamp': vehicle['attributes']['updated_at'],
                        'stop_sequence': vehicle['attributes']['current_stop_sequence'],
                        'trip_id': vehicle['relationships']['trip']['data']['id'],
                        'route_id': vehicle['relationships']['route']['data']['id']
                    }
                    mbta_fields.append(row)
                df = pd.DataFrame(mbta_fields)
                if df.empty:
                    continue
                df['agency'] = 'MBTA'
                df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
                mbta_dfs.append(df)
            if mbta_dfs:
                mbta_df = pd.concat(mbta_dfs, ignore_index=True)
                mbta_df.to_csv(mbta_file, mode='a', header=not os.path.exists(mbta_file), index=False)
        except Exception as e:
            print(f"MBTA error: {e}")

# MTA
        try:
            mta_dfs = []
            for route in mta_routes:
                mta_params = {"key": MTA_KEY, "LineRef": f"MTA NYCT_{route}"}  # needs to be in the loop
                response = requests.get(mta_url, params=mta_params)
                delivery = response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]
                mta_vehicles = delivery.get('VehicleActivity', [])  # empty list if no active vehicles
                mta_rows = []
                for vehicle in mta_vehicles:
                    mvj = vehicle['MonitoredVehicleJourney']  # the path to data
                    if 'MonitoredCall' not in mvj:  # skip buses in layover
                        continue
                    mc = mvj['MonitoredCall']  # the path to data
                    if 'ExpectedArrivalTime' not in mc:  # skip vehicles with no prediction
                        continue
                    # parse both times so it can be subtracted to get dev
                    aimed = pd.to_datetime(mc['AimedArrivalTime'])
                    expected = pd.to_datetime(mc['ExpectedArrivalTime'])
                    row = {
                        'vehicle_id': mvj['VehicleRef'].replace('MTA NYCT_', ''),  # removing prefix
                        'lat': mvj['VehicleLocation']['Latitude'],
                        'lon': mvj['VehicleLocation']['Longitude'],
                        'direction': int(mvj['DirectionRef']),  # making the string "0"/"1" to int
                        'timestamp': pd.to_datetime(vehicle['RecordedAtTime']),
                        'route_id': mvj['LineRef'].replace('MTA NYCT_', ''),  # removing prefix
                        'call_distance': mc['Extensions']['Distances']['CallDistanceAlongRoute'],  # stop_sequence equivalent
                        'deviation': (expected - aimed).total_seconds() / 60  # positive = late, negative = early
                    }
                    mta_rows.append(row)
                df = pd.DataFrame(mta_rows)
                if df.empty:  # skip if no valid vehicles for this route
                    continue
                df['agency'] = 'MTA'
                df['timestamp'] = df['timestamp'].dt.tz_localize(None)  # making timezone to match WMATA/MBTA
                mta_dfs.append(df)
            if mta_dfs:
                mta_df = pd.concat(mta_dfs, ignore_index=True)
                mta_df.to_csv(mta_file, mode='a', header=not os.path.exists(mta_file), index=False)
        except Exception as e:
            print(f"MTA error: {e}")

        poll_count += 1
        print(f"Poll complete: {pd.Timestamp.now().strftime('%H:%M:%S')} (poll {poll_count})")
        
        if poll_count % 60 == 0: # This will allow me to auto commit to github in both repos (Have to specify exact path or wont work)
            try:
                os.system('jupyter nbconvert --to notebook --inplace "api_script.ipynb"') # auto save notebook
                os.system(f'git add data/ "api_script.ipynb" && git commit -m "auto: update data poll {poll_count}" && git push origin main')
                os.system('mkdir -p /Users/ahmadalexander/Desktop/python/eco590/assignment-1-Ahmad-Alexander/data/')
                os.system('cp -r /Users/ahmadalexander/Desktop/gtfs_reliability_analysis/scripts/python/data/ /Users/ahmadalexander/Desktop/python/eco590/assignment-1-Ahmad-Alexander/data/')
                os.system('cp /Users/ahmadalexander/Desktop/gtfs_reliability_analysis/scripts/python/api_script.ipynb /Users/ahmadalexander/Desktop/python/eco590/assignment-1-Ahmad-Alexander/')
                os.system(f'cd /Users/ahmadalexander/Desktop/python/eco590/assignment-1-Ahmad-Alexander && git add data/ api_script.ipynb && git commit -m "auto: update data poll {poll_count}" && git push origin master')
                print(f"Pushed to GitHub at poll {poll_count}")
            except Exception as e:
                print(f"Git push error: {e}")
            
        time.sleep(60)  # wait 1 minute before next poll

    except KeyboardInterrupt:
        print("Polling stopped.")
        break

Poll complete: 23:38:13 (poll 1)
Poll complete: 23:39:19 (poll 2)
Poll complete: 23:40:25 (poll 3)
Poll complete: 23:41:31 (poll 4)
Poll complete: 23:42:37 (poll 5)
Poll complete: 23:43:43 (poll 6)
Poll complete: 23:44:49 (poll 7)
Poll complete: 23:45:54 (poll 8)
Poll complete: 23:47:00 (poll 9)
Poll complete: 23:48:05 (poll 10)
Poll complete: 23:49:10 (poll 11)
Poll complete: 23:50:16 (poll 12)
Poll complete: 23:51:22 (poll 13)
Poll complete: 23:52:27 (poll 14)
Poll complete: 23:53:33 (poll 15)
Poll complete: 23:54:39 (poll 16)
Poll complete: 23:55:46 (poll 17)
Poll complete: 23:56:51 (poll 18)
Poll complete: 23:57:57 (poll 19)
Poll complete: 23:59:02 (poll 20)
Poll complete: 00:00:08 (poll 21)
Poll complete: 00:01:13 (poll 22)
Poll complete: 00:02:19 (poll 23)
Poll complete: 00:03:24 (poll 24)
Poll complete: 00:04:30 (poll 25)
Poll complete: 00:05:37 (poll 26)
Poll complete: 00:06:42 (poll 27)
Poll complete: 00:07:48 (poll 28)
Poll complete: 00:08:54 (poll 29)
Poll complete: 00:10:00

[NbConvertApp] Converting notebook api_script.ipynb to notebook
[NbConvertApp] Writing 14551 bytes to api_script.ipynb


[main 74182ca] auto: update data poll 60
 4 files changed, 7952 insertions(+), 103 deletions(-)


To github.com:Ahmad-Alexander/gtfs_reliability_analysis.git
   8342f68..74182ca  main -> main


[master 5d548bb] auto: update data poll 60
 4 files changed, 7963 insertions(+), 219 deletions(-)


...

[main 92d7be3] auto: update data poll 1920
 4 files changed, 17257 insertions(+), 1 deletion(-)


To github.com:Ahmad-Alexander/gtfs_reliability_analysis.git
   a8cbccb..92d7be3  main -> main


[master a7e33fd] auto: update data poll 1920
 4 files changed, 17257 insertions(+), 1 deletion(-)


To github.com:ProfKaltenberg/assignment-1-Ahmad-Alexander.git
   6661cd8..a7e33fd  master -> master


Pushed to GitHub at poll 1920
Poll complete: 11:07:33 (poll 1921)


## Resources

- Standardizing time zones:

https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.tz_localize.html

- If continue and break statements:

https://www.educative.io/answers/what-are-break-continue-and-pass-statements-in-python?utm_campaign=Pmax_feb25&utm_source=google&utm_medium=ppc&utm_content=&utm_term=&eid=5082902844932096&utm_term=&utm_campaign=%5BMar+25%5D+Pmax.+-+Coding+Interview+Prep&utm_source=adwords&utm_medium=ppc&hsa_acc=5451446008&hsa_cam=22344713166&hsa_grp=&hsa_ad=&hsa_src=x&hsa_tgt=&hsa_kw=&hsa_mt=&hsa_net=adwords&hsa_ver=3&gad_source=1&gad_campaignid=22354833079&gbraid=0AAAAADfWLuSny8RMeOlF5ThZ04eXKwptZ&gclid=CjwKCAjwvqjOBhAGEiwAngeQnVWlwZMFRhHCXD6ekkGarMTrSQa1YzFdylNc3-Rritp4jo-j1r1iWRoCCMoQAvD_BwE

- Timing Loops

https://www.digitalocean.com/community/tutorials/python-time-sleep

- Try Except

https://www.w3schools.com/python/python_try_except.asp
https://www.reddit.com/r/learnpython/comments/1blfl9e/explain_try_except_structure_in_practical_examples/
https://stackoverflow.com/questions/62062226/how-to-work-with-try-and-except-in-python
https://docs.python.org/3/tutorial/errors.html   
 https://stackoverflow.com/questions/21120947/catching-keyboardinterrupt-in-python-during-program-shutdown

- Auto Commit

https://docs.github.com/en/authentication/connecting-to-github-with-ssh
 https://docs.github.com/en/authentication/connecting-to-github-with-ssh/generating-a-new-ssh-key-and-adding-it-to-the-ssh-agent
 https://discourse.jupyter.org/t/how-to-run-a-notebook-using-command-line/3475
 https://stackoverflow.com/questions/5620525/git-pushing-to-two-repos-in-one-command
 https://stackoverflow.com/questions/4255865/git-push-to-multiple-repositories-simultaneously
 